In [3]:
import sys
!{sys.executable} -m pip install torch torchvision torchaudio

   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.3/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.3/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.8/124.1 MB 970.4 kB/s eta 0:02:08
   ---------------------------------------- 1.3/124.1 MB 1.4 MB/s eta 0:01:26
    --------------------------------------- 2.1/124.1 MB 1.9 MB/s eta 0:01:05
    --------------------------------------- 2.9/124.1 MB 2.2 MB/s eta 0:00:54
   - -------------------------------------- 3.7/124.1 MB 2.5 MB/s eta 0:00:49
   - -------------------------------------- 4.5/124.1 MB 2.7 MB/s eta 0:00:45
   - -------------------------------------- 5.2/124.1 MB 2.8 MB/s eta 0:00:43
   - ---------------------

In [1]:
import re
import random
import unicodedata
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import gensim.downloader as api

from torch.utils.data import Dataset, DataLoader
from torch.nn import Module
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import classification_report, f1_score
from torch.utils.data import Dataset, DataLoader




In [2]:
data=pd.read_csv('aratox_train.tsv', sep='\t')

In [3]:
len(data)

33038

In [4]:
characters = []
for i in range(0, len(data)):
    text = data['Text'][i]
    words = text.split()
    for word in words:
        for char in word:
            characters.append(char)

characters = list(set(characters))
print(sorted(characters))
print(len(characters))

['.', 'A', 'B', 'C', 'E', 'L', 'N', 'O', 'Z', 'a', 'c', 'e', 'f', 'g', 'i', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'y', '¤', '¬', '®', '°', '¿', '×', 'ç', 'é', 'ī', 'İ', 'ɵ', 'ˬ', '̈', '̥', '̷', '͜', '͡', 'ׇ', '،', '؏', '؛', '\u061c', '؟', 'ء', 'آ', 'أ', 'ؤ', 'إ', 'ئ', 'ا', 'ب', 'ة', 'ت', 'ث', 'ج', 'ح', 'خ', 'د', 'ذ', 'ر', 'ز', 'س', 'ش', 'ص', 'ض', 'ط', 'ظ', 'ع', 'غ', 'ـ', 'ف', 'ق', 'ك', 'ل', 'م', 'ن', 'ه', 'و', 'ى', 'ي', 'ً', 'ٓ', 'ٗ', '٪', '٬', 'ٱ', 'پ', 'چ', 'ژ', 'ڤ', 'ڨ', 'ک', 'ڪ', 'گ', 'ڵ', 'ھ', 'ہ', 'ۈ', 'ۋ', 'ی', 'ے', 'ۖ', 'ۗ', 'ۚ', 'ۛ', '\u06dd', '۞', 'ۡ', 'ۣ', 'ۦ', '۽', 'ฟ', 'ไ', 'ღ', 'ᴗ', '᷄', '᷅', '\u200b', '\u200c', '\u200d', '–', '—', '‘', '’', '“', '”', '‼', '⁃', '⁉', '\u2066', '\u2067', '\u2069', '⁽', '⃣', '℡', '←', '↓', '↗', '↘', '↙', '↩', '↴', '⇉', '⇓', '⇣', 'Ⓜ', '┉', '┊', '▪', '▫', '◀', '●', '◾', '☀', '☁', '☂', '☆', '☕', '☘', '☝', '☪', '☮', '☹', '☺', '☻', '♀', '♂', '♠', '♡', '♥', '♦', '♨', '♻', '⚁', '⚔', '⚘', '⚜', '⚠', '⚡', '⚧', '⚪', '⚫', '⛄', '⛔', '✂', '✅', '✊', '✋',

***preprocessing***

In [5]:
import re
import unicodedata

def clean_text(text):
    text = str(text)

    # Unicode normalization
    text = unicodedata.normalize('NFKC', text)

    # Remove English characters
    text = re.sub(r'[A-Za-z]', '', text)

    # Normalize Arabic characters
    text = re.sub(r'[إأآٱ]', 'ا', text)
    text = re.sub(r'[ىی]', 'ي', text)
    text = re.sub(r'[ؤئ]', 'ء', text)
    text = re.sub(r'[کڪﻙﻚﻛﻜ]', 'ك', text)
    text = re.sub(r'[ھہ]', 'ه', text)

    # Remove Arabic diacritics
    text = re.sub(
        r'[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]',
        '',
        text
    )

    # Remove Tatweel
    text = re.sub(r'ـ+', '', text)

    text = re.sub(r'[ے؏۽]', '', text)
    # Remove invisible/control characters
    text = re.sub(
        r'[\u061C\u200B-\u200F\u202A-\u202E\u2060-\u206F\uFEFF]',
        '',
        text
    )

    # Keep Arabic characters, numbers, whitespace and useful punctuation
    text = re.sub(
        r'[^\u0600-\u06FF\u0750-\u077F0-9\s!?؟،؛٪٬]',
        '',
        text
    )

    # Reduce repeated punctuation
    text = re.sub(r'([!?؟،؛])\1+', r'\1', text)

    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

data['Text'] = data['Text'].apply(clean_text)

In [6]:
characters = []
for i in range(0, len(data)):
    text = data['Text'][i]
    words = text.split()
    for word in words:
        for char in word:
            characters.append(char)

characters = list(set(characters))
print(sorted(characters))
print(len(characters))

['!', '?', '،', '؛', '؟', 'ء', 'ا', 'ب', 'ة', 'ت', 'ث', 'ج', 'ح', 'خ', 'د', 'ذ', 'ر', 'ز', 'س', 'ش', 'ص', 'ض', 'ط', 'ظ', 'ع', 'غ', 'ف', 'ق', 'ك', 'ل', 'م', 'ن', 'ه', 'و', 'ي', '٪', '٬', 'پ', 'چ', 'ژ', 'ڤ', 'ڨ', 'گ', 'ڵ', 'ۈ', 'ۋ']
46


In [7]:
data=data[['Text', 'Cussing', 'Hatred', 'Appearance', 'Racial','Sexual','Violence','NOT']]

In [8]:
train_data, valid_data = train_test_split(data, test_size=0.2, random_state=42)

In [9]:
X_train=train_data['Text']
y_train=train_data[['Cussing', 'Hatred', 'Appearance', 'Racial','Sexual','Violence','NOT']]

In [10]:
print(X_train.shape)

(26430,)


In [11]:
X_val=valid_data['Text']
y_val=valid_data[['Cussing', 'Hatred', 'Appearance', 'Racial','Sexual','Violence','NOT']]

In [12]:
labels = [
    'Cussing',
    'Hatred',
    'Appearance',
    'Racial',
    'Sexual',
    'Violence',
    'NOT'
]

In [14]:
tfidf = TfidfVectorizer(
    analyzer='word',
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)
# analyzer='word' means that we are using words as features, ngram_range=(1,2) means that we are using unigrams and bigrams as features, min_df=2 means that we are ignoring features that appear in less than 2 documents, max_df=0.9 means that we are ignoring features that appear in more than 90% of the documents, sublinear_tf=True means that we are using sublinear term frequency scaling = 1+log(tf).

# The TF-IDF vector is per document, where each feature represents the TF-IDF score of a word/ngram in that document based on its frequency across all documents. If max_features is limited, we keep only the features with the highest overall term frequency.
X_train_tfidf = tfidf.fit_transform(X_train)
X_valid_tfidf = tfidf.transform(X_val)

In [15]:
svm = MultiOutputClassifier(
    SVC(
        kernel='linear',
        C=1.0
    )
)
# We are using MultiOutputClassifier because we have multiple labels for each text.
svm.fit(X_train_tfidf, y_train)



,estimator,SVC(kernel='linear')
,n_jobs,None
,C,1.0
,kernel,'linear'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200


In [16]:

y_pred = svm.predict(X_valid_tfidf)



print("Macro F1:", f1_score(y_val, y_pred, average='macro'))

print(
    classification_report(
        y_val,
        y_pred,
        target_names=labels,
        zero_division=0
    )
)

Macro F1: 0.8812734586583046
              precision    recall  f1-score   support

     Cussing       0.98      0.90      0.94      1985
      Hatred       0.91      0.78      0.84      1294
  Appearance       0.97      0.82      0.89       911
      Racial       0.91      0.64      0.75       831
      Sexual       0.99      0.95      0.97      1481
    Violence       0.97      0.72      0.82       565
         NOT       0.96      0.97      0.96      1838

   micro avg       0.96      0.86      0.91      8905
   macro avg       0.95      0.82      0.88      8905
weighted avg       0.96      0.86      0.90      8905
 samples avg       0.92      0.89      0.90      8905



In [17]:

tfidf_char = TfidfVectorizer(
    analyzer='char',
    ngram_range=(2, 5),
    min_df=2,
    max_df=0.9,
    sublinear_tf=True
)

X_train_char = tfidf_char.fit_transform(X_train)
X_valid_char = tfidf_char.transform(X_val)

svm_char = MultiOutputClassifier(
    SVC(kernel='linear', C=1.0)
)

svm_char.fit(X_train_char, y_train)



,estimator,SVC(kernel='linear')
,n_jobs,None
,C,1.0
,kernel,'linear'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200


In [18]:
y_pred_char = svm_char.predict(X_valid_char)



print("Macro F1:", f1_score(y_val, y_pred, average='macro'))

print(
    classification_report(
        y_val,
        y_pred,
        target_names=labels,
        zero_division=0
    )
)

Macro F1: 0.8812734586583046
              precision    recall  f1-score   support

     Cussing       0.98      0.90      0.94      1985
      Hatred       0.91      0.78      0.84      1294
  Appearance       0.97      0.82      0.89       911
      Racial       0.91      0.64      0.75       831
      Sexual       0.99      0.95      0.97      1481
    Violence       0.97      0.72      0.82       565
         NOT       0.96      0.97      0.96      1838

   micro avg       0.96      0.86      0.91      8905
   macro avg       0.95      0.82      0.88      8905
weighted avg       0.96      0.86      0.90      8905
 samples avg       0.92      0.89      0.90      8905



In [18]:
# X_train_char.shape

In [19]:
# X_train_tfidf.shape

In [20]:
def tokenize(text):
    return text.split()


train_tokens = [tokenize(t) for t in X_train]
val_tokens = [tokenize(t) for t in X_val]


In [21]:
def build_vocab(tokenized_texts, min_count=2):
    counter = Counter()
    for toks in tokenized_texts:
        counter.update(toks)
    vocab = {"<pad>": 0, "<unk>": 1}
    for word, count in counter.items():
        if count >= min_count:
            vocab[word] = len(vocab)
    return vocab, counter

In [22]:
vocab, counter = build_vocab(train_tokens, min_count=2)
V = len(vocab)
print(V)

28645


In [23]:

vocab, counter = build_vocab(train_tokens, min_count=2)
V = len(vocab)


train_data_filtered = [
    (tokens, label)
    for tokens, label in zip(
        train_tokens,
        y_train[labels].values
    )
    if len(tokens) > 0
]

val_data_filtered = [
    (tokens, label)
    for tokens, label in zip(
        val_tokens,
        y_val[labels].values
    )
    if len(tokens) > 0
]

train_tokens = [x[0] for x in train_data_filtered]
train_labels = [x[1] for x in train_data_filtered]

val_tokens = [x[0] for x in val_data_filtered]
val_labels = [x[1] for x in val_data_filtered]


In [30]:
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class TextDataset(Dataset):
    def __init__(self, texts, labels, vocab):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        tokens = self.texts[idx]

        ids = [
            self.vocab.get(word, self.vocab["<unk>"])
            for word in tokens
        ]

        return (
            torch.tensor(ids, dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.float32)
        )


def collate_fn(batch):
    sequences, labels = zip(*batch)

    lengths = torch.tensor(
        [len(x) for x in sequences],
        dtype=torch.long
    )

    sequences = torch.nn.utils.rnn.pad_sequence(
        sequences,
        batch_first=True,#!!
        padding_value=vocab["<pad>"]
    )

    labels = torch.stack(labels)#it converts the list of tensors into a single tensor by stacking them along a new dimension, which is necessary for batch processing in PyTorch.

    return sequences, lengths, labels


class LSTMClassifier(Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_dim,
        num_classes,
        num_layers=2,
        dropout=0.5
    ):
        super().__init__()

        self.embedding = torch.nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=vocab["<pad>"]
        )

        self.embedding_dropout = torch.nn.Dropout(0.3)

        self.lstm = torch.nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.attn_fc = torch.nn.Linear(hidden_dim * 2, hidden_dim)
        self.attn_v = torch.nn.Linear(hidden_dim, 1, bias=False)

        self.dropout = torch.nn.Dropout(dropout)

        self.layer_norm = torch.nn.LayerNorm(hidden_dim * 2)

        self.fc1 = torch.nn.Linear(hidden_dim * 2, hidden_dim * 2)

        self.fc2 = torch.nn.Linear(hidden_dim * 2, hidden_dim)

        self.fc3 = torch.nn.Linear(hidden_dim, num_classes)

    def forward(self, x, lengths):

        x = self.embedding(x)

        x = self.embedding_dropout(x)
        #pack_padded_sequence is used to handle variable-length sequences in RNNs. It tell the model the sequece have different real lengths and don't waste your time on padding tokens. It is important to sort the sequences by length in descending order before packing, but since we set enforce_sorted=False, it will handle unsorted sequences as well.batch_first=true means that the input tensor has the shape (batch_size, seq_len, embedding_dim) instead of (seq_len, batch_size, embedding_dim).
        packed = torch.nn.utils.rnn.pack_padded_sequence(
            x,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )
        #packed contains a batch_sizes tells the LSTM how many sequences are present at each time step, allowing it to efficiently process variable-length sequences without wasting computation on padding tokens.
        #its look like [4,4,4,3,3,2,1] this means at time step 0, there are 4 sequences,
        #we need this because the LSTM works in parallel across the batch dimension, and it needs to know how many sequences are active at each time step to avoid processing padding tokens. The LSTM will only process the active sequences at each time step, which improves efficiency and prevents the model from learning from padding tokens.
        #packed.data.shape give the total words at once like [485,200] but batch_sizes give the number of sequences at each time step like [4,4,4,3,3,2,1] 
       
        #packed_output contains the output of the LSTM for each time step, while hidden and cell contain the final hidden and cell states for each layer of the LSTM. The output is packed because the input sequences were packed, which allows the LSTM to efficiently process variable-length sequences without wasting computation on padding tokens.
        #if i dont have an attention layer i can use the last hidden state of the LSTM as the representation of the entire sequence, but since we have an attention layer, we will use the attention mechanism to compute a weighted sum of the LSTM outputs, which allows the model to focus on different parts of the sequence when making predictions.
        packed_output, (hidden, cell) = self.lstm(packed)
        output, _ = torch.nn.utils.rnn.pad_packed_sequence(
            packed_output,
            batch_first=True
        )

        seq_len = output.size(1)
        #unsqueeze only add a dimension of size 1 at the specified position
        mask = torch.arange(seq_len, device=output.device).unsqueeze(0) < lengths.unsqueeze(1).to(output.device)
        #mask is a boolean tensor that indicates which elements in the output tensor correspond to valid time steps (True) and which correspond to padding (False).
        energy = torch.tanh(self.attn_fc(output))
        #e=tanh(W*h+b) where W is the weight matrix of the attention layer, h is the hidden state of the LSTM at each time step, and b is the bias term. The tanh activation function is applied to introduce non-linearity and help the model learn complex relationships in the data.
        #(Batch_size, seq_len, hidden_dim * 2) -> (Batch_size, seq_len, hidden_dim)
        scores = self.attn_v(energy).squeeze(-1)
        #s=W*e where W is the weight matrix of the attention layer and e is the energy tensor. The squeeze(-1) operation removes the last dimension of size 1 from the scores tensor, resulting in a shape of (Batch_size, seq_len). This gives us a score for each time step in the sequence, which will be used to compute attention weights.
        #(Batch_size,seq_len,hidden_dim)->(Batch_size,seq_len)
        scores = scores.masked_fill(~mask, float("-inf"))
        #give scores of -inf to the padding tokens so that they don't contribute to the attention weights.
        attn_weights = torch.softmax(scores, dim=1)
        #softmax is applied over the sequence dimension to convert the scores into probabilities that sum to 1 for each sequence in the batch.
        #7eta kda ta2ked fehm dlwa2ty el data btegy (Batch_size, seq_len, hidden_dim * 2) ba7awelha le (Batch_size, seq_len) we ba3den softmax we bta3 fa kol word fel sentence bta5od probability bta3to 3la 7asb el attention weights fa kda mesh far2a 7agm el sentence.
        attn_out = torch.bmm(attn_weights.unsqueeze(1), output).squeeze(1)
        #attn_weights.unsqueeze(1) makes it from (Batch_size, seq_len) to (Batch_size, 1, seq_len) so that we can do batch matrix multiplication with output which is (Batch_size, seq_len, hidden_dim * 2). The result is (Batch_size, 1, hidden_dim * 2) which we then squeeze to (Batch_size, hidden_dim * 2).
        masked_output = output.masked_fill(~mask.unsqueeze(-1), float("-inf"))
        #~mask.unsqueeze(-1) adds a dimension of size 1 (Batch_size, seq_len, 1) to the mask tensor so that it can be broadcasted to the shape of the output tensor (Batch_size, seq_len, hidden_dim * 2). This allows us to apply the mask to the output tensor, setting the values of the padding tokens to negative infinity.
        #this is made to make max pooling not a part of attention layer
       

        combined = self.layer_norm(attn_out)

        x = self.dropout(combined)

        x = torch.relu(self.fc1(x))

        x = self.dropout(x)

        x = torch.relu(self.fc2(x))

        x = self.dropout(x)

        return self.fc3(x)


train_dataset = TextDataset(
    train_tokens,
    train_labels,
    vocab
)

val_dataset = TextDataset(
    val_tokens,
    val_labels,
    vocab
)


train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn
)


model = LSTMClassifier(
    vocab_size=V,
    embedding_dim=200,
    hidden_dim=128,
    num_classes=len(labels),
    num_layers=2,
    dropout=0.5
).to(DEVICE)


train_labels_array = np.array(train_labels)

pos_counts = train_labels_array.sum(axis=0)

neg_counts = train_labels_array.shape[0] - pos_counts

pos_weight = torch.tensor(
    neg_counts / np.clip(pos_counts, 1, None),
    dtype=torch.float32
).to(DEVICE)


criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
#this is loss function and contains sigmoid in it so no need for it .


optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)


EPOCHS = 25

best_val_f1_macro = 0.0
best_model_state = None
patience_counter = 0
early_stop_patience = 6

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    train_correct = 0
    train_total = 0

    for x, lengths, y in train_loader:

        x = x.to(DEVICE)
        lengths = lengths.to(DEVICE)
        y = y.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(x, lengths)

        loss = criterion(outputs, y)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        total_loss += loss.item()

        probabilities = torch.sigmoid(outputs)

        predictions = (probabilities >= 0.5).float()

        train_correct += (predictions == y).sum().item()
        train_total += y.numel()

    train_accuracy = train_correct / train_total

    model.eval()

    val_correct = 0
    val_total = 0
    val_loss = 0

    all_predictions = []
    all_targets = []

    with torch.no_grad():

        for x, lengths, y in val_loader:

            x = x.to(DEVICE)
            lengths = lengths.to(DEVICE)
            y = y.to(DEVICE)

            outputs = model(x, lengths)

            loss = criterion(outputs, y)

            val_loss += loss.item()

            probabilities = torch.sigmoid(outputs)

            predictions = (probabilities >= 0.5).float()

            val_correct += (predictions == y).sum().item()
            val_total += y.numel()

            all_predictions.append(
                predictions.cpu().numpy()
            )

            all_targets.append(
                y.cpu().numpy()
            )

    val_accuracy = val_correct / val_total

    all_predictions = np.vstack(all_predictions)
    all_targets = np.vstack(all_targets)

    val_f1_micro = f1_score(
        all_targets,
        all_predictions,
        average="micro",
        zero_division=0
    )

    val_f1_macro = f1_score(
        all_targets,
        all_predictions,
        average="macro",
        zero_division=0
    )

    scheduler.step(val_f1_macro)

    if val_f1_macro > best_val_f1_macro:
        best_val_f1_macro = val_f1_macro
        best_model_state = {
            k: v.clone() for k, v in model.state_dict().items()
        }
        patience_counter = 0
    else:
        patience_counter += 1

    print(
        f"Epoch {epoch+1}/{EPOCHS} "
        f"Loss: {total_loss/len(train_loader):.4f} "
        f"Train Acc: {train_accuracy:.4f} "
        f"Val Loss: {val_loss/len(val_loader):.4f} "
        f"Val Acc: {val_accuracy:.4f} "
        f"Micro F1: {val_f1_micro:.4f} "
        f"Macro F1: {val_f1_macro:.4f}"
    )

    if patience_counter >= early_stop_patience:
        break

model.load_state_dict(best_model_state)

Epoch 1/25 Loss: 0.7732 Train Acc: 0.7684 Val Loss: 0.5130 Val Acc: 0.8823 Micro F1: 0.7389 Macro F1: 0.7250
Epoch 2/25 Loss: 0.5065 Train Acc: 0.8768 Val Loss: 0.4075 Val Acc: 0.9134 Micro F1: 0.7983 Macro F1: 0.7734
Epoch 3/25 Loss: 0.4179 Train Acc: 0.9050 Val Loss: 0.3929 Val Acc: 0.9272 Micro F1: 0.8250 Macro F1: 0.7947
Epoch 4/25 Loss: 0.3635 Train Acc: 0.9199 Val Loss: 0.3642 Val Acc: 0.9294 Micro F1: 0.8318 Macro F1: 0.8120
Epoch 5/25 Loss: 0.3227 Train Acc: 0.9311 Val Loss: 0.3635 Val Acc: 0.9389 Micro F1: 0.8508 Macro F1: 0.8259
Epoch 6/25 Loss: 0.2927 Train Acc: 0.9392 Val Loss: 0.4136 Val Acc: 0.9464 Micro F1: 0.8657 Macro F1: 0.8421
Epoch 7/25 Loss: 0.2676 Train Acc: 0.9453 Val Loss: 0.4764 Val Acc: 0.9481 Micro F1: 0.8696 Macro F1: 0.8464
Epoch 8/25 Loss: 0.2487 Train Acc: 0.9510 Val Loss: 0.4141 Val Acc: 0.9456 Micro F1: 0.8656 Macro F1: 0.8428
Epoch 9/25 Loss: 0.2256 Train Acc: 0.9551 Val Loss: 0.4360 Val Acc: 0.9470 Micro F1: 0.8685 Macro F1: 0.8484
Epoch 10/25 Loss: 0

<All keys matched successfully>

In [157]:
model.eval()

def predict(text_tokens):

    ids = [
        vocab.get(word, vocab["<unk>"])
        for word in text_tokens
    ]

    x = torch.tensor(
        [ids],
        dtype=torch.long
    ).to(DEVICE)

    lengths = torch.tensor(
        [len(ids)],
        dtype=torch.long
    )

    with torch.no_grad():

        outputs = model(x, lengths)

        probabilities = torch.sigmoid(outputs)

        predictions = (probabilities >= 0.5).float()

    probabilities = probabilities[0].cpu().numpy()
    predictions = predictions[0].cpu().numpy()

    result = [int(predictions[i]) for i in range(len(labels))]


    return result

In [160]:
all_predictions = np.vstack(all_predictions)
all_targets = np.vstack(all_targets)

print(
    classification_report(
        all_targets,
        all_predictions,
        target_names=labels,
        zero_division=0
    )
)

val_f1_micro = f1_score(
    all_targets,
    all_predictions,
    average="micro",
    zero_division=0
)

val_f1_macro = f1_score(
    all_targets,
    all_predictions,
    average="macro",
    zero_division=0
)

print(f"Micro F1: {val_f1_micro:.4f}")
print(f"Macro F1: {val_f1_macro:.4f}")

              precision    recall  f1-score   support

     Cussing       0.96      0.92      0.94      1985
      Hatred       0.79      0.86      0.82      1294
  Appearance       0.86      0.91      0.88       911
      Racial       0.71      0.76      0.74       831
      Sexual       0.97      0.96      0.97      1481
    Violence       0.84      0.81      0.83       565
         NOT       0.93      0.98      0.95      1838

   micro avg       0.89      0.91      0.90      8905
   macro avg       0.87      0.89      0.88      8905
weighted avg       0.89      0.91      0.90      8905
 samples avg       0.92      0.93      0.91      8905

Micro F1: 0.8972
Macro F1: 0.8754


In [24]:
from gensim.models import Word2Vec

SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EMBEDDING_PATH = "full_uni_cbow_300_twitter/full_uni_cbow_300_twitter.mdl"

w2v = Word2Vec.load(EMBEDDING_PATH)
wv = w2v.wv
EMBEDDING_DIM = wv.vector_size

vocab = {"<pad>": 0, "<unk>": 1}
for i, word in enumerate(wv.index_to_key):
    vocab[word] = i + 2

V = len(vocab)

embedding_matrix = np.zeros((V, EMBEDDING_DIM), dtype=np.float32)
embedding_matrix[1] = wv.vectors.mean(axis=0)#the <unk>token is initialized with the mean of all word vectors in the pre-trained embedding. 
embedding_matrix[2:] = wv.vectors#the remaining tokens are initialized with their corresponding pre-trained word vectors from the Word2Vec model. 

class TextDataset(Dataset):
    def __init__(self, texts, labels, vocab):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        tokens = self.texts[idx]

        ids = [
            self.vocab.get(word, self.vocab["<unk>"])
            for word in tokens
        ]

        return (
            torch.tensor(ids, dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.float32)
        )


def collate_fn(batch):
    sequences, labels = zip(*batch)

    lengths = torch.tensor(
        [len(x) for x in sequences],
        dtype=torch.long
    )

    sequences = torch.nn.utils.rnn.pad_sequence(
        sequences,
        batch_first=True,
        padding_value=vocab["<pad>"]
    )

    labels = torch.stack(labels)

    return sequences, lengths, labels


class LSTMClassifier(Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_dim,
        num_classes,
        num_layers=2,
        dropout=0.5
    ):
        super().__init__()

        self.embedding = torch.nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=vocab["<pad>"]
        )

        self.embedding_dropout = torch.nn.Dropout(0.3)

        self.lstm = torch.nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.attn_fc = torch.nn.Linear(hidden_dim * 2, hidden_dim)
        self.attn_v = torch.nn.Linear(hidden_dim, 1, bias=False)

        self.dropout = torch.nn.Dropout(dropout)

        self.layer_norm = torch.nn.LayerNorm(hidden_dim * 2)

        self.fc1 = torch.nn.Linear(hidden_dim * 2, hidden_dim * 2)

        self.fc2 = torch.nn.Linear(hidden_dim * 2, hidden_dim)

        self.fc3 = torch.nn.Linear(hidden_dim, num_classes)

    def forward(self, x, lengths):

        x = self.embedding(x)

        x = self.embedding_dropout(x)

        packed = torch.nn.utils.rnn.pack_padded_sequence(
            x,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        packed_output, (hidden, cell) = self.lstm(packed)
        output, _ = torch.nn.utils.rnn.pad_packed_sequence(
            packed_output,
            batch_first=True
        )

        seq_len = output.size(1)

        mask = torch.arange(seq_len, device=output.device).unsqueeze(0) < lengths.unsqueeze(1).to(output.device)

        energy = torch.tanh(self.attn_fc(output))

        scores = self.attn_v(energy).squeeze(-1)

        scores = scores.masked_fill(~mask, float("-inf"))

        attn_weights = torch.softmax(scores, dim=1)

        attn_out = torch.bmm(attn_weights.unsqueeze(1), output).squeeze(1)

        masked_output = output.masked_fill(~mask.unsqueeze(-1), float("-inf"))

        combined = self.layer_norm(attn_out)

        x = self.dropout(combined)

        x = torch.relu(self.fc1(x))

        x = self.dropout(x)

        x = torch.relu(self.fc2(x))

        x = self.dropout(x)

        return self.fc3(x)


train_dataset = TextDataset(
    train_tokens,
    train_labels,
    vocab
)

val_dataset = TextDataset(
    val_tokens,
    val_labels,
    vocab
)


train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn
)


model = LSTMClassifier(
    vocab_size=V,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=128,
    num_classes=len(labels),
    num_layers=2,
    dropout=0.5
).to(DEVICE)

model.embedding.weight.data.copy_(torch.from_numpy(embedding_matrix))


train_labels_array = np.array(train_labels)

pos_counts = train_labels_array.sum(axis=0)

neg_counts = train_labels_array.shape[0] - pos_counts

pos_weight = torch.tensor(
    neg_counts / np.clip(pos_counts, 1, None),
    dtype=torch.float32
).to(DEVICE)


criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)


optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)


EPOCHS = 25

best_val_f1_macro = 0.0
best_model_state = None
patience_counter = 0
early_stop_patience = 6

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    train_correct = 0
    train_total = 0

    for x, lengths, y in train_loader:

        x = x.to(DEVICE)
        lengths = lengths.to(DEVICE)
        y = y.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(x, lengths)

        loss = criterion(outputs, y)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        total_loss += loss.item()

        probabilities = torch.sigmoid(outputs)

        predictions = (probabilities >= 0.5).float()

        train_correct += (predictions == y).sum().item()
        train_total += y.numel()

    train_accuracy = train_correct / train_total

    model.eval()

    val_correct = 0
    val_total = 0
    val_loss = 0

    all_predictions = []
    all_targets = []

    with torch.no_grad():

        for x, lengths, y in val_loader:

            x = x.to(DEVICE)
            lengths = lengths.to(DEVICE)
            y = y.to(DEVICE)

            outputs = model(x, lengths)

            loss = criterion(outputs, y)

            val_loss += loss.item()

            probabilities = torch.sigmoid(outputs)

            predictions = (probabilities >= 0.5).float()

            val_correct += (predictions == y).sum().item()
            val_total += y.numel()

            all_predictions.append(
                predictions.cpu().numpy()
            )

            all_targets.append(
                y.cpu().numpy()
            )

    val_accuracy = val_correct / val_total

    all_predictions = np.vstack(all_predictions)
    all_targets = np.vstack(all_targets)

    val_f1_micro = f1_score(
        all_targets,
        all_predictions,
        average="micro",
        zero_division=0
    )

    val_f1_macro = f1_score(
        all_targets,
        all_predictions,
        average="macro",
        zero_division=0
    )

    scheduler.step(val_f1_macro)

    if val_f1_macro > best_val_f1_macro:
        best_val_f1_macro = val_f1_macro
        best_model_state = {
            k: v.clone() for k, v in model.state_dict().items()
        }
        patience_counter = 0
    else:
        patience_counter += 1

    print(
        f"Epoch {epoch+1}/{EPOCHS} "
        f"Loss: {total_loss/len(train_loader):.4f} "
        f"Train Acc: {train_accuracy:.4f} "
        f"Val Loss: {val_loss/len(val_loader):.4f} "
        f"Val Acc: {val_accuracy:.4f} "
        f"Micro F1: {val_f1_micro:.4f} "
        f"Macro F1: {val_f1_macro:.4f}"
    )

    if patience_counter >= early_stop_patience:
        break

model.load_state_dict(best_model_state)

Epoch 1/25 Loss: 0.5830 Train Acc: 0.8340 Val Loss: 0.4022 Val Acc: 0.9196 Micro F1: 0.8083 Macro F1: 0.7912
Epoch 2/25 Loss: 0.4087 Train Acc: 0.9060 Val Loss: 0.3432 Val Acc: 0.9263 Micro F1: 0.8274 Macro F1: 0.8121
Epoch 3/25 Loss: 0.3422 Train Acc: 0.9267 Val Loss: 0.3361 Val Acc: 0.9415 Micro F1: 0.8580 Macro F1: 0.8431
Epoch 4/25 Loss: 0.2913 Train Acc: 0.9401 Val Loss: 0.3496 Val Acc: 0.9488 Micro F1: 0.8729 Macro F1: 0.8546
Epoch 5/25 Loss: 0.2569 Train Acc: 0.9479 Val Loss: 0.3828 Val Acc: 0.9521 Micro F1: 0.8798 Macro F1: 0.8605
Epoch 6/25 Loss: 0.2227 Train Acc: 0.9556 Val Loss: 0.3642 Val Acc: 0.9487 Micro F1: 0.8732 Macro F1: 0.8553
Epoch 7/25 Loss: 0.2018 Train Acc: 0.9609 Val Loss: 0.4277 Val Acc: 0.9489 Micro F1: 0.8739 Macro F1: 0.8548
Epoch 8/25 Loss: 0.1764 Train Acc: 0.9658 Val Loss: 0.4300 Val Acc: 0.9479 Micro F1: 0.8717 Macro F1: 0.8503
Epoch 9/25 Loss: 0.1484 Train Acc: 0.9721 Val Loss: 0.4389 Val Acc: 0.9524 Micro F1: 0.8817 Macro F1: 0.8645
Epoch 10/25 Loss: 0

<All keys matched successfully>

In [ ]:
import os
import time
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn import Module
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

os.environ["PYTHONHASHSEED"] = str(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if DEVICE.type == "cpu":
    print(
        "WARNING: No GPU detected. Fine-tuning a transformer on CPU is "
        "extremely slow. Use a GPU (e.g. Google Colab, free tier) if you "
        "want epochs to finish in minutes instead of hours."
    )

LABEL_COLUMNS = ["Cussing", "Hatred", "Appearance", "Racial", "Sexual", "Violence", "NOT"]

MODEL_NAME = "aubmindlab/bert-base-arabertv02"
MAX_LEN = 200                 # shorter sequences = much faster (was 128)
BATCH_SIZE = 32               # larger batch = fewer steps (was 16)
EPOCHS = 25                    # enough to see fine-tuning work (was 25)
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 1e-2
EARLY_STOP_PATIENCE = 3       # (was 6)
WARMUP_RATIO = 0.06

# Only fine-tune the last N transformer layers of AraBERT; freeze the rest.
# This is the biggest speed lever: it massively cuts the backward-pass cost
# while still letting the model adapt to your task. Set to 0 to freeze all
# of BERT (fastest, "feature extraction" style), or a large number
# (e.g. 999) to fine-tune the whole model like the original script.
UNFREEZE_LAST_N_LAYERS = 2

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts.reset_index(drop=True)
        self.labels = labels.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts.iloc[idx]

        if not isinstance(text, str) or len(text.strip()) == 0:
            text = tokenizer.unk_token if tokenizer.unk_token is not None else "."

        label = self.labels.iloc[idx].values.astype(np.float32)

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_tensors="pt"
        )
        #attention_mask is a tensor that tells the model which tokens in the sequence are real and which are just padding filler.

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.float32)
        }


class AraBERTClassifier(Module):
    def __init__(self, model_name, num_classes, dropout=0.3, unfreeze_last_n=2):
        super().__init__()

        self.bert = AutoModel.from_pretrained(model_name)

        hidden_size = self.bert.config.hidden_size

        self._freeze_backbone(unfreeze_last_n)

        self.dropout = torch.nn.Dropout(dropout)

        self.fc1 = torch.nn.Linear(hidden_size, hidden_size)

        self.layer_norm = torch.nn.LayerNorm(hidden_size)

        self.fc2 = torch.nn.Linear(hidden_size, num_classes)

    def _freeze_backbone(self, unfreeze_last_n):
        # Freeze everything first.
        for param in self.bert.parameters():
            param.requires_grad = False

        if unfreeze_last_n <= 0:
            return

        # Unfreeze the pooler (if present).
        if hasattr(self.bert, "pooler") and self.bert.pooler is not None:
            for param in self.bert.pooler.parameters():
                param.requires_grad = True

        # Unfreeze the last N encoder layers.
        encoder_layers = self.bert.encoder.layer
        n = min(unfreeze_last_n, len(encoder_layers))

        for layer in encoder_layers[-n:]:
            for param in layer.parameters():
                param.requires_grad = True

        trainable = sum(p.numel() for p in self.bert.parameters() if p.requires_grad)
        total = sum(p.numel() for p in self.bert.parameters())
        print(f"BERT backbone: {trainable:,} / {total:,} params trainable "
              f"(last {n} layer(s) unfrozen)")

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        if hasattr(outputs, "pooler_output") and outputs.pooler_output is not None:
            pooled = outputs.pooler_output
        else:
            last_hidden = outputs.last_hidden_state

            mask = attention_mask.unsqueeze(-1).float()

            summed = (last_hidden * mask).sum(dim=1)

            counts = mask.sum(dim=1).clamp(min=1e-9)

            pooled = summed / counts

        x = self.dropout(pooled)

        x = torch.relu(self.fc1(x))

        x = self.layer_norm(x)

        x = self.dropout(x)

        return self.fc2(x)


def build_dataloaders(X_train, y_train, X_val, y_val):
    train_dataset = TextDataset(X_train, y_train, tokenizer, MAX_LEN)
    val_dataset = TextDataset(X_val, y_val, tokenizer, MAX_LEN)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )

    return train_loader, val_loader


def compute_pos_weight(y_train):
    train_labels_array = np.array(y_train)

    pos_counts = train_labels_array.sum(axis=0)

    neg_counts = train_labels_array.shape[0] - pos_counts

    pos_weight = torch.tensor(
        neg_counts / np.clip(pos_counts, 1, None),
        dtype=torch.float32
    ).to(DEVICE)

    return pos_weight


def find_best_thresholds(all_targets, all_probs):
    num_classes = all_targets.shape[1]

    best_thresholds = np.full(num_classes, 0.5, dtype=np.float32)

    candidate_thresholds = np.arange(0.1, 0.9, 0.02)

    for i in range(num_classes):
        best_f1 = -1.0
        best_t = 0.5

        for t in candidate_thresholds:
            preds = (all_probs[:, i] >= t).astype(np.float32)

            f1 = f1_score(all_targets[:, i], preds, zero_division=0)

            if f1 > best_f1:
                best_f1 = f1
                best_t = t

        best_thresholds[i] = best_t

    return best_thresholds


def evaluate(model, val_loader, criterion, thresholds=None):
    model.eval()

    val_loss = 0.0

    all_probs = []
    all_targets = []

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            outputs = model(input_ids, attention_mask)

            loss = criterion(outputs, labels)

            val_loss += loss.item()

            probabilities = torch.sigmoid(outputs)

            all_probs.append(probabilities.cpu().numpy())
            all_targets.append(labels.cpu().numpy())

    all_probs = np.vstack(all_probs)
    all_targets = np.vstack(all_targets)

    if thresholds is None:
        predictions = (all_probs >= 0.5).astype(np.float32)
    else:
        predictions = (all_probs >= thresholds.reshape(1, -1)).astype(np.float32)

    val_f1_micro = f1_score(all_targets, predictions, average="micro", zero_division=0)
    val_f1_macro = f1_score(all_targets, predictions, average="macro", zero_division=0)

    avg_val_loss = val_loss / max(len(val_loader), 1)

    return avg_val_loss, val_f1_micro, val_f1_macro, all_probs, all_targets


def train(X_train, y_train, X_val, y_val):
    train_loader, val_loader = build_dataloaders(X_train, y_train, X_val, y_val)

    model = AraBERTClassifier(
        MODEL_NAME,
        num_classes=len(LABEL_COLUMNS),
        unfreeze_last_n=UNFREEZE_LAST_N_LAYERS
    ).to(DEVICE)

    pos_weight = compute_pos_weight(y_train)

    criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    # Only pass trainable parameters to the optimizer — frozen params don't
    # need gradients tracked or updated, which saves memory and time.
    trainable_params = [p for p in model.parameters() if p.requires_grad]

    optimizer = torch.optim.AdamW(
        trainable_params,
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    total_steps = len(train_loader) * EPOCHS

    warmup_steps = int(total_steps * WARMUP_RATIO)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    use_amp = DEVICE.type == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    best_val_f1_macro = 0.0
    best_model_state = None
    best_thresholds = None
    patience_counter = 0

    for epoch in range(EPOCHS):
        epoch_start = time.time()

        model.train()

        total_loss = 0.0

        for batch in train_loader:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            optimizer.zero_grad()

            with torch.cuda.amp.autocast(enabled=use_amp):
                outputs = model(input_ids, attention_mask)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()

            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)

            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            total_loss += loss.item()

        avg_train_loss = total_loss / max(len(train_loader), 1)

        val_loss, val_f1_micro, val_f1_macro, all_probs, all_targets = evaluate(
            model, val_loader, criterion
        )

        epoch_time = time.time() - epoch_start

        print(
            f"Epoch {epoch + 1}/{EPOCHS} "
            f"({epoch_time:.1f}s) "
            f"Train Loss: {avg_train_loss:.4f} "
            f"Val Loss: {val_loss:.4f} "
            f"Micro F1: {val_f1_micro:.4f} "
            f"Macro F1: {val_f1_macro:.4f}"
        )

        if val_f1_macro > best_val_f1_macro:
            best_val_f1_macro = val_f1_macro

            best_model_state = {
                k: v.clone() for k, v in model.state_dict().items()
            }

            best_thresholds = find_best_thresholds(all_targets, all_probs)

            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"Early stopping at epoch {epoch + 1}")
            break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    return model, best_thresholds


if __name__ == "__main__":
    model, best_thresholds = train(X_train, y_train, X_val, y_val)

    train_loader, val_loader = build_dataloaders(X_train, y_train, X_val, y_val)

    criterion = torch.nn.BCEWithLogitsLoss()

    val_loss, val_f1_micro, val_f1_macro, all_probs, all_targets = evaluate(
        model, val_loader, criterion, thresholds=best_thresholds
    )

    print(f"Final Val Loss: {val_loss:.4f}")
    print(f"Final Micro F1 (tuned thresholds): {val_f1_micro:.4f}")
    print(f"Final Macro F1 (tuned thresholds): {val_f1_macro:.4f}")

    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "thresholds": best_thresholds,
            "label_columns": LABEL_COLUMNS,
            "model_name": MODEL_NAME,
            "max_len": MAX_LEN
        },
        "arabert_classifier.pt"
    )

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
C:\Users\zeyad\AppData\Local\Temp\ipykernel_24544\637784038.py:300: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_am

BERT backbone: 14,766,336 / 135,193,344 params trainable (last 2 layer(s) unfrozen)


KeyboardInterrupt: 